In [1]:
### basic prompt usage
from langchain_openai import ChatOpenAI
import os
from dotenv import load_dotenv

load_dotenv()

model = ChatOpenAI(
    model="meta/muse-glimmer-30b",
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=os.getenv("NVIDIA_API_KEY"),
    temperature=0.2,
    max_tokens=500,
    reasoning_effort="low"
)
#response = model.invoke("hey what are you what can you do and when was your last update")
messages = [
    {"role": "system", "content": "you are a pro translator translate what i gave you into french"},
    {"role": "user", "content": "hello my friend"},
    ]
response = model.invoke(messages)
print(response.content)


Bonjour mon ami


In [20]:
### template prompt usage + chains
from langchain_core.prompts import ChatPromptTemplate
template=ChatPromptTemplate.from_messages([
    {"role":"system","content":"you are a pro translator translate what i gave you into {language}"},
    {"role":"user","content":"{query}"}
])

prompt=template.invoke({
    "query":"hello, my friend how are you",
    "language":"french"
    }
)
response = model.invoke(prompt)
print(response.content)

Bonjour, mon ami, comment vas-tu ?


In [26]:
### create pipes

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

template=ChatPromptTemplate.from_messages([
    {"role":"system","content":"you are a pro translator translate what i gave you into {language}"},
    {"role":"user","content":"{query}"}
])

pipe= template | model | StrOutputParser()

response = pipe.invoke({
    "query":"hello",
    "language":"french"
    }
)

print(response)

Bonjour


In [ ]:
### llm + hostory 
history=[]

while True:
    query=input(">")
    if query == "exit":
        break
    history.append({"role":"user","content":query})
    response = model.invoke(history)
    history.append({"role":"assistant","content":response.content})
    print(response.content)

 

Hello! 👋 How can I help you today?
Ginger cats are the best! 🧡

They're often super affectionate, playful, and have that classic "marmalade" charm. A lot of people say orange tabbies are especially confident and chatty — basically little sunbeams with paws.

Is yours a tabby/orange tabby, or more of a solid ginger? What's his/her name and personality like?
Your cat is ginger — that classic orange/marmalade color you mentioned! 🧡


In [ ]:
### structereed_output
from pydantic import BaseModel,Field,ValidationError
from typing import List

class User(BaseModel):
    name: str
    age: int

class Movie (BaseModel):
    title: str = Field(max_length=100)
    year: int
    type: str
    
class Movies(BaseModel):
    movies: List[Movie]
    
try:
   result = model.with_structured_output(Movies,method="function_calling").invoke(
    "I need 3 movies: action, adventure, and mystery, "
    "released after 2014 and before 2024. with good rating"
)
   print(result.model_dump()) ## view on the outputed structure
   for movie in result.movies:
       print(f"{movie.title} ({movie.year}) - {movie.type}")
except ValidationError as e:
    print(e)



Mad Max: Fury Road (2015) - action
Jumanji: Welcome to the Jungle (2017) - adventure
Knives Out (2019) - mystery


In [43]:
### streaming 
res = model.stream('what are llms')
for chunk in res:
    print(chunk.content,end="")


LLMs are **Large Language Models**.

They are a type of artificial intelligence model trained on massive amounts of text data to understand, generate, and predict human language.

**How they work, in simple terms:**

1. **Training on data:** An LLM is trained on trillions of words from books, websites, code, articles, etc. It doesn't memorize the text, it learns statistical patterns of how words and sentences relate to each other.
2. **Next-word prediction:** At its core, an LLM is a very advanced next-word predictor. Given a prompt like "The cat sat on the...", it predicts the most likely next words based on patterns it learned.
3. **Scale matters:** The "Large" part refers to the size. LLMs have billions to trillions of parameters, which are the internal knobs the model adjusts during training to capture language patterns. More parameters + more data = better understanding and generation.

**What they can do:**

* **Generate text:** Write essays, emails, stories, code, summaries
* **